In [103]:
pip install xlsxwriter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [149]:
import pandas as pd

In [150]:
cashbook_raw = pd.read_excel("Bank Reconciliation Sample.xlsx", sheet_name="Company Cashbook", skiprows=4)
bank_raw = pd.read_excel("Bank Reconciliation Sample.xlsx", sheet_name="Bank Statement", skiprows=5)

In [151]:
# Reshaping Company Cashbook sheet
import pandas as pd

def reshape_cashbook(path):
    """
    Read and reshape the Company Cashbook sheet into tidy format:
    Date | Details | Amount | Type
    """

    # Debit section = first three columns
    debit = cashbook_raw.iloc[:, [1, 2, 3]].copy()
    debit.columns = ["Date", "Details", "Amount"]
    debit["Type"] = "Debit"

    # Credit section = next three columns
    credit = cashbook_raw.iloc[:, [5, 6, 7]].copy()
    credit.columns = ["Date", "Details", "Amount"]
    credit["Type"] = "Credit"

    # Combine both
    cashbook = pd.concat([debit, credit], ignore_index=True)

    # Normalize
    cashbook["Date"] = pd.to_datetime(cashbook["Date"], errors="coerce")
    cashbook["Amount"] = pd.to_numeric(cashbook["Amount"], errors="coerce")

    return cashbook

In [152]:
# reshaping Bank Statement sheet

bank_raw.columns = bank_raw.columns.str.strip()
#print(bank_raw.columns.tolist())

def reshape_bank(bank_raw):
    """
    Reshape Bank Statement into tidy format:
    Date | Details | Amount | Type
    """

    # Debit side (money going out)
    debit = bank_raw[["Date", "Particulars", "Debit"]].copy()
    debit.columns = ["Date", "Details", "Amount"]
    debit["Type"] = "Debit"

    # Credit side (money coming in)
    credit = bank_raw[["Date", "Particulars", "Credit"]].copy()
    credit.columns = ["Date", "Details", "Amount"]
    credit["Type"] = "Credit"

    # Combine both
    bank_stmt = pd.concat([debit, credit], ignore_index=True)

    # Normalize
    bank_stmt["Date"] = pd.to_datetime(bank_stmt["Date"], errors="coerce")
    bank_stmt["Amount"] = pd.to_numeric(bank_stmt["Amount"], errors="coerce")

    return bank_stmt

In [153]:
# Load the raw sheet
#cashbook_raw = pd.read_excel("Bank_Reconciliation_Sample.xlsx", sheet_name="Company Cashbook", skiprows=4)

# Call the function
cashbook = reshape_cashbook(cashbook_raw)

bank_stmt = reshape_bank(bank_raw)

# Now 'cashbook' is ready for reconciliation
cashbook.head(20)
bank_stmt.head(20)

,Date,Details,Amount,Type
0,2023-06-01,Opening Balance,NaN,Debit
1,2023-06-05,Electricity Bill Cheque # 0007864,24300.0,Debit
2,2023-06-05,Dividend Income - Online Transfer,NaN,Debit
3,2023-06-05,Receipt Cheque # 0007981,NaN,Debit
4,2023-06-08,Payment Cheque # 0007866,17400.0,Debit
5,2023-06-10,Outward Payment Chaeque # 0007867,1700.0,Debit
6,2023-06-13,Receipt Cheque # 07982,NaN,Debit
7,2023-06-14,Bank Charges,3200.0,Debit
8,2023-06-20,Payment Cheque # 0007865,30700.0,Debit
9,2023-06-20,Receipt Cheque # 07983,NaN,Debit


In [154]:
cashbook.head(20)

,Date,Details,Amount,Type
0,2023-06-01,Balance b/d,186200.0,Debit
1,2023-06-04,Mr. Ali - 7981,21200.0,Debit
2,2023-06-09,Mr.Younis - 7982,18500.0,Debit
3,2023-06-19,Mrs. Samia - 7983,11800.0,Debit
4,2023-06-24,Mr. Arif - 7984,4700.0,Debit
5,2023-06-27,Mrs. Ayesha - 7985,27900.0,Debit
6,2023-06-29,Mrs. Husna - 7986,9800.0,Debit
7,2023-06-30,Mr. Arshad - 7987,13400.0,Debit
8,NaT,NaN,NaN,Debit
9,NaT,NaN,NaN,Debit


In [155]:
bank_stmt.head(18)

,Date,Details,Amount,Type
0,2023-06-01,Opening Balance,NaN,Debit
1,2023-06-05,Electricity Bill Cheque # 0007864,24300.0,Debit
2,2023-06-05,Dividend Income - Online Transfer,NaN,Debit
3,2023-06-05,Receipt Cheque # 0007981,NaN,Debit
4,2023-06-08,Payment Cheque # 0007866,17400.0,Debit
5,2023-06-10,Outward Payment Chaeque # 0007867,1700.0,Debit
6,2023-06-13,Receipt Cheque # 07982,NaN,Debit
7,2023-06-14,Bank Charges,3200.0,Debit
8,2023-06-20,Payment Cheque # 0007865,30700.0,Debit
9,2023-06-20,Receipt Cheque # 07983,NaN,Debit


In [156]:
cashbook_raw.columns

Index(['Unnamed: 0', 'Date', 'Details', 'Amount ($)', 'Unnamed: 4', 'Date.1',
       'Details.1', 'Amount ($).1'],
      dtype='str')

In [157]:

import re
import pandas as pd

def extract_cheque_no(detail):
    #Extract cheque/reference number.Example:'Electricity Bill Cheque #0007864' returns 7864
    
    nums = re.findall(r"\d+", str(detail))
    
    if nums:
        return str(int(nums[-1]))
    return None

def clean_dataframe(df):
    #Remove Balance and Total rows.
    
    df = df[
        ~df["Details"].astype(str).str.contains(
            "Balance|Total",
            case=False,
            na=False
        )]

    return df    

def normalize_data(df):
    df = df[~df["Details"].astype(str).str.contains("Balance|Total", case=False, na=False)]
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    df["Details"] = df["Details"].astype(str).str.strip()
    df["ChequeNo"] = df["Details"].apply(extract_cheque_no)
    return df


def reconcile_to_excel(cashbook, bank_stmt, output_path="Final_Reconciliation.xlsx"):

    cashbook = normalize_data(cashbook)
    bank_stmt = normalize_data(bank_stmt)

    # Reverse the effect because bank and cashbook have opposite entries
    cashbook["BankType"] = cashbook["Type"].map({
        "Debit": "Credit",
        "Credit": "Debit"
    })

    # Matched transactions
    matched = cashbook.merge(
        bank_stmt,
        left_on=["Amount", "ChequeNo", "BankType"],
        right_on=["Amount", "ChequeNo", "Type"],
        how="inner",
        suffixes=("_Cashbook", "_Bank")
    )

    # Create matching keys
    
    cashbook["MatchKey"] = list(zip(cashbook["Amount"], cashbook["ChequeNo"], cashbook["Type"]))
    bank_stmt["MatchKey"] = list(zip(bank_stmt["Amount"], bank_stmt["ChequeNo"], bank_stmt["Type"]))

    matched_cash_keys = set(zip(
        matched["Amount"],
        matched["ChequeNo"],
        matched["Type_Cashbook"]
    ))

    matched_bank_keys = set(zip(
        matched["Amount"],
        matched["ChequeNo"],
        matched["Type_Bank"]
    ))

    # W-2 : Cashbook entries not in bank
    w2 = cashbook[~cashbook["MatchKey"].isin(matched_cash_keys)]

    # W-1 : Bank entries not in cashbook
    w1 = bank_stmt[~bank_stmt["MatchKey"].isin(matched_bank_keys)]

    # Duplicates
    cashbook_dupes = cashbook[
        cashbook.duplicated(
            subset=["Date", "Amount", "ChequeNo"],
            keep=False
        )
    ]

    bank_dupes = bank_stmt[
        bank_stmt.duplicated(
            subset=["Date", "Amount", "ChequeNo"],
            keep=False
        )
    ]

    # Balances
    cash_debit = cashbook.loc[cashbook["Type"] == "Debit", "Amount"].sum()
    cash_credit = cashbook.loc[cashbook["Type"] == "Credit", "Amount"].sum()

    bank_debit = bank_stmt.loc[bank_stmt["Type"] == "Debit", "Amount"].sum()
    bank_credit = bank_stmt.loc[bank_stmt["Type"] == "Credit", "Amount"].sum()

    cash_balance = cash_debit - cash_credit
    bank_balance = bank_credit - bank_debit

    summary = pd.DataFrame({
        "Metric": [
            "Cashbook Balance",
            "Bank Balance",
            "Matched Transactions",
            "Bank Only Items (W1)",
            "Cashbook Only Items (W2)"
        ],
        "Value": [
            cash_balance,
            bank_balance,
            len(matched),
            w1["Amount"].sum(),
            w2["Amount"].sum()
        ]
    })

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        cashbook.to_excel(writer, sheet_name="Cashbook", index=False)
        bank_stmt.to_excel(writer, sheet_name="Bank Statement", index=False)
        matched.to_excel(writer, sheet_name="Matched", index=False)
        w1.to_excel(writer, sheet_name="W1 Bank Only", index=False)
        w2.to_excel(writer, sheet_name="W2 Cashbook Only", index=False)
        cashbook_dupes.to_excel(writer, sheet_name="Cashbook Duplicates", index=False)
        bank_dupes.to_excel(writer, sheet_name="Bank Duplicates", index=False)
        summary.to_excel(writer, sheet_name="Summary", index=False)

    return output_path

In [158]:

reconcile_to_excel(cashbook, bank_stmt, "Final_Reconciliation.xlsx")

'Final_Reconciliation.xlsx'